In [ ]:
import os
import json
import pandas as pd
import numpy as np
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_text
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from langchain_community.vectorstores import FAISS
from langchain.embeddings.tensorflow import TensorflowHubEmbeddings
import kagglehub
from tqdm.auto import tqdm
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize
from transformers import TFAutoModelForSeq2SeqLM, AutoTokenizer

# Download NLTK resources
nltk.download('punkt')
nltk.download('stopwords')

class CORD19Agent:
    def __init__(self, data_path=None, model_path='./models'):
        """Initialize the CORD-19 agent with paths for data and models."""
        self.data_path = data_path
        self.model_path = model_path
        self.metadata = None
        self.vector_store = None
        self.embedding_model = None
        self.summarizer_model = None
        self.summarizer_tokenizer = None
        
        # Create model directory if it doesn't exist
        os.makedirs(model_path, exist_ok=True)
        
        # Initialize embedding model for vector storage
        print("Loading Universal Sentence Encoder...")
        self.embedding_model = hub.load("https://tfhub.dev/google/universal-sentence-encoder-multilingual/3")
        
        # Initialize summarization model
        print("Loading T5 summarization model...")
        self.summarizer_tokenizer = AutoTokenizer.from_pretrained("t5-small")
        self.summarizer_model = TFAutoModelForSeq2SeqLM.from_pretrained("t5-small")
    
    def download_dataset(self):
        """Download the CORD-19 dataset from Kaggle."""
        print("Downloading CORD-19 dataset...")
        self.data_path = kagglehub.dataset_download("googleai/dataset-metadata-for-cord19")
        print(f"Dataset downloaded to: {self.data_path}")
        return self.data_path
    
    def load_metadata(self, file_path=None):
        """Load metadata from the CORD-19 dataset."""
        if file_path is None:
            # Use file in downloaded dataset path
            file_path = os.path.join(self.data_path, 'metadata.csv')
        
        print(f"Loading metadata from {file_path}...")
        self.metadata = pd.read_csv(file_path)
        print(f"Loaded {len(self.metadata)} records")
        
        # Preview the data structure
        print("\nMetadata preview:")
        print(self.metadata.head())
        print("\nColumns:", self.metadata.columns.tolist())
        
        return self.metadata
    
    def preprocess_text(self, text):
        """Clean and preprocess text for better embedding and summarization."""
        if pd.isna(text):
            return ""
        
        # Convert to lowercase
        text = text.lower()
        
        # Remove special characters and digits
        text = re.sub(r'[^\w\s]', '', text)
        text = re.sub(r'\d+', '', text)
        
        # Remove extra whitespace
        text = re.sub(r'\s+', ' ', text).strip()
        
        return text
    
    def create_document_embeddings(self, batch_size=64):
        """Create embeddings for documents using the Universal Sentence Encoder."""
        if self.metadata is None:
            raise ValueError("Metadata not loaded. Call load_metadata() first.")
        
        print("Creating document embeddings...")
        
        # Combine title and abstract for more meaningful embeddings
        documents = []
        for _, row in tqdm(self.metadata.iterrows(), total=len(self.metadata)):
            title = "" if pd.isna(row.get('title')) else row.get('title')
            abstract = "" if pd.isna(row.get('abstract')) else row.get('abstract')
            
            # Combine with appropriate handling of missing values
            doc_text = f"{title}. {abstract}" if title and abstract else title or abstract or ""
            doc_text = self.preprocess_text(doc_text)
            
            if doc_text.strip():  # Only add non-empty documents
                documents.append({
                    'id': row.get('cord_uid', ''),
                    'text': doc_text,
                    'title': title,
                    'abstract': abstract
                })
        
        print(f"Processing {len(documents)} documents with text content")
        
        # Create embeddings in batches to avoid memory issues
        all_embeddings = []
        for i in tqdm(range(0, len(documents), batch_size)):
            batch = documents[i:i+batch_size]
            batch_texts = [doc['text'] for doc in batch]
            batch_embeddings = self.embedding_model(batch_texts).numpy()
            all_embeddings.append(batch_embeddings)
            
        # Concatenate all batches
        if all_embeddings:
            embeddings = np.vstack(all_embeddings)
            print(f"Created embeddings with shape: {embeddings.shape}")
            
            # Create a LangChain-compatible embedding function
            embed_func = lambda x: self.embedding_model(x).numpy().tolist()
            langchain_embeddings = TensorflowHubEmbeddings(embed_func)
            
            # Create vector store
            texts = [doc['text'] for doc in documents]
            metadatas = [{'id': doc['id'], 'title': doc['title']} for doc in documents]
            
            print("Creating FAISS vector store...")
            self.vector_store = FAISS.from_texts(texts, langchain_embeddings, metadatas=metadatas)
            
            # Save the vector store
            self.vector_store.save_local("cord19_vector_store")
            print("Vector store saved to 'cord19_vector_store'")
            
            return documents, embeddings
        else:
            print("No embeddings created, check your data")
            return [], None
    
    def extract_keywords(self, documents, num_topics=5, num_words=10):
        """Extract keywords using Non-negative Matrix Factorization on TF-IDF vectors."""
        print("Extracting keywords using NMF...")
        
        # Extract text from documents
        texts = [doc['text'] for doc in documents if doc['text'].strip()]
        
        # Create TF-IDF matrix
        tfidf_vectorizer = TfidfVectorizer(
            max_df=0.95, 
            min_df=2,
            stop_words='english',
            max_features=10000
        )
        
        tfidf_matrix = tfidf_vectorizer.fit_transform(texts)
        
        # Apply NMF to extract topics
        nmf_model = NMF(n_components=num_topics, random_state=42)
        nmf_model.fit(tfidf_matrix)
        
        # Get feature names (words)
        feature_names = tfidf_vectorizer.get_feature_names_out()
        
        # Extract top words for each topic
        topics = []
        for topic_idx, topic in enumerate(nmf_model.components_):
            top_words_idx = topic.argsort()[:-num_words-1:-1]
            top_words = [feature_names[i] for i in top_words_idx]
            topics.append({
                'topic_id': topic_idx,
                'words': top_words
            })
        
        return topics
    
    def generate_summary(self, text, max_length=150):
        """Generate a summary of the given text using T5."""
        if not text or pd.isna(text) or len(text.strip()) == 0:
            return "No text provided for summarization."
        
        # Prepare the input text
        input_text = "summarize: " + text
        
        # Tokenize the input
        inputs = self.summarizer_tokenizer(input_text, return_tensors="tf", 
                                         max_length=512, truncation=True)
        
        # Generate summary
        summary_ids = self.summarizer_model.generate(
            inputs["input_ids"],
            max_length=max_length,
            num_beams=4,
            early_stopping=True
        )
        
        # Decode the summary
        summary = self.summarizer_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
        
        return summary
    
    def generate_batch_summaries(self, documents, sample_size=10):
        """Generate summaries for a batch of documents."""
        print(f"Generating summaries for {sample_size} sample documents...")
        
        # Select a sample of documents for summarization
        if sample_size < len(documents):
            import random
            sample_docs = random.sample(documents, sample_size)
        else:
            sample_docs = documents
        
        summaries = []
        for doc in tqdm(sample_docs):
            # Combine title and abstract for summarization
            title = doc.get('title', '')
            abstract = doc.get('abstract', '')
            
            if not abstract:
                summary = "No abstract available for summarization."
            else:
                # Generate summary from abstract
                summary = self.generate_summary(abstract)
            
            summaries.append({
                'id': doc.get('id', ''),
                'title': title,
                'summary': summary
            })
        
        return summaries
    
    def search_similar_documents(self, query, k=5):
        """Search for documents similar to the query using vector similarity."""
        if self.vector_store is None:
            try:
                print("Loading existing vector store...")
                embed_func = lambda x: self.embedding_model(x).numpy().tolist()
                langchain_embeddings = TensorflowHubEmbeddings(embed_func)
                self.vector_store = FAISS.load_local("cord19_vector_store", langchain_embeddings)
            except Exception as e:
                print(f"Error loading vector store: {e}")
                print("Please run create_document_embeddings() first")
                return []
        
        # Search for similar documents
        results = self.vector_store.similarity_search_with_score(query, k=k)
        
        # Format results
        similar_docs = []
        for doc, score in results:
            similar_docs.append({
                'content': doc.page_content,
                'metadata': doc.metadata,
                'similarity_score': float(score)
            })
        
        return similar_docs

    def run_pipeline(self, query=None, sample_size=10):
        """Run the complete pipeline from data loading to summarization and search."""
        # Step 1: Download dataset if not already available
        if not self.data_path:
            self.download_dataset()
        
        # Step 2: Load metadata
        self.load_metadata()
        
        # Step 3: Create document embeddings and vector store
        documents, _ = self.create_document_embeddings()
        
        # Step 4: Extract keywords/topics
        topics = self.extract_keywords(documents)
        print("\nExtracted Topics/Keywords:")
        for topic in topics:
            print(f"Topic {topic['topic_id']}: {', '.join(topic['words'])}")
        
        # Step 5: Generate summaries for sample documents
        summaries = self.generate_batch_summaries(documents, sample_size)
        print("\nGenerated Summaries:")
        for i, summary in enumerate(summaries[:3]):  # Show first 3 for brevity
            print(f"\n--- Document {i+1} ---")
            print(f"Title: {summary['title']}")
            print(f"Summary: {summary['summary']}")
        
        # Step 6: Perform a similarity search if query provided
        if query:
            print(f"\nSearching for documents similar to: '{query}'")
            similar_docs = self.search_similar_documents(query)
            print("\nSimilar Documents:")
            for i, doc in enumerate(similar_docs):
                print(f"\n--- Match {i+1} (Score: {doc['similarity_score']:.4f}) ---")
                print(f"Title: {doc['metadata'].get('title', 'N/A')}")
                print(f"Content: {doc['content'][:200]}...")
        
        return {
            "topics": topics,
            "summaries": summaries,
            "similar_docs": similar_docs if query else None
        }

# Example usage
if __name__ == "__main__":
    # Initialize the agent
    agent = CORD19Agent()
    
    # Run the complete pipeline with a search query
    results = agent.run_pipeline(
        query="covid-19 transmission in indoor environments",
        sample_size=5
    )